# Dataset v3: sampler ablation

Сравнение четырех способов формирования train batches при неизменных LRW features, BiGRU, validation и seeds. Test split не загружается.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from collections import Counter
from pathlib import Path
import math
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler


def load_torch_file(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
cache_dir = drive_root / 'controlled_lrw_frontend_bigru_dataset_v3'
output_dir = drive_root / 'controlled_lrw_dataset_v3_sampler_ablation'
output_dir.mkdir(parents=True, exist_ok=True)
runs_path = output_dir / 'dataset_v3_sampler_ablation_runs.csv'

train_cache = load_torch_file(cache_dir / 'dataset_v3_train_lrw_resnet18_features.pt')
val_cache = load_torch_file(cache_dir / 'dataset_v3_val_lrw_resnet18_features.pt')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

vocab = [
    'есть', 'когда', 'чтобы', 'просто', 'будет',
    'может', 'значит', 'время', 'потом', 'человек',
]
word_to_idx = {word: index for index, word in enumerate(vocab)}

assert train_cache['features'].shape == (1901, 24, 512)
assert val_cache['features'].shape == (105, 24, 512)
assert set(train_cache['speakers']).isdisjoint(set(val_cache['speakers']))
print('device:', device)
print('train:', train_cache['features'].shape)
print('validation:', val_cache['features'].shape)
print('TEST SPLIT IS NOT LOADED.')


In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


class FeatureSequenceDataset(Dataset):
    def __init__(self, cache):
        self.features = cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in cache['words']], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def make_train_loader(strategy, seed):
    dataset = FeatureSequenceDataset(train_cache)
    generator = torch.Generator().manual_seed(seed)

    if strategy == 'uniform_shuffle':
        return DataLoader(
            dataset, batch_size=32, shuffle=True, generator=generator, num_workers=0
        )

    if strategy == 'word_inverse':
        keys = list(train_cache['words'])
        exponent = 1.0
    elif strategy == 'speaker_word_inverse':
        keys = list(zip(train_cache['speakers'], train_cache['words']))
        exponent = 1.0
    elif strategy == 'speaker_word_sqrt':
        keys = list(zip(train_cache['speakers'], train_cache['words']))
        exponent = 0.5
    else:
        raise ValueError(f'Unknown strategy: {strategy}')

    counts = Counter(keys)
    weights = torch.tensor(
        [1.0 / (counts[key] ** exponent) for key in keys], dtype=torch.double
    )
    sampler = WeightedRandomSampler(
        weights,
        num_samples=len(weights),
        replacement=True,
        generator=generator,
    )
    return DataLoader(dataset, batch_size=32, sampler=sampler, num_workers=0)


val_loader = DataLoader(
    FeatureSequenceDataset(val_cache), batch_size=32, shuffle=False, num_workers=0
)


def evaluate(model, criterion):
    model.eval()
    total_loss = 0.0
    total = 0
    predictions = []
    labels = []
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            total_loss += criterion(outputs, batch_y).item() * batch_y.size(0)
            total += batch_y.size(0)
            predictions.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(batch_y.cpu().tolist())
    return {
        'loss': total_loss / total,
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'predicted_classes': len(set(predictions)),
    }


In [ ]:
def run_one_seed(strategy, seed, num_epochs=30, patience=7):
    set_reproducible_seed(seed)
    train_loader = make_train_loader(strategy, seed)
    set_reproducible_seed(seed)

    model = BiGRUClassifier(len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_macro_f1 = -1.0
    best_val_loss = float('inf')
    best_row = None
    epochs_without_improvement = 0
    start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * batch_y.size(0)
            train_correct += (outputs.argmax(dim=1) == batch_y).sum().item()
            train_total += batch_y.size(0)

        metrics = evaluate(model, criterion)
        improved = (
            metrics['macro_f1'] > best_macro_f1
            or (metrics['macro_f1'] == best_macro_f1 and metrics['loss'] < best_val_loss)
        )
        if improved:
            best_macro_f1 = metrics['macro_f1']
            best_val_loss = metrics['loss']
            epochs_without_improvement = 0
            best_row = {
                'strategy': strategy,
                'seed': seed,
                'best_epoch': epoch,
                'train_accuracy_at_best': train_correct / train_total,
                'train_loss_at_best': train_loss_sum / train_total,
                'validation_loss': metrics['loss'],
                'validation_accuracy': metrics['accuracy'],
                'validation_macro_f1': metrics['macro_f1'],
                'validation_balanced_accuracy': metrics['balanced_accuracy'],
                'predicted_classes': metrics['predicted_classes'],
            }
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    best_row['epochs_completed'] = epoch
    best_row['training_seconds'] = time.perf_counter() - start
    print(
        f"{strategy:>22} seed={seed:4d} | "
        f"macro-F1={best_row['validation_macro_f1']:.4f} | "
        f"acc={best_row['validation_accuracy']:.4f} | "
        f"{best_row['training_seconds']:.1f}s"
    )
    return best_row


In [ ]:
STRATEGIES = [
    'speaker_word_inverse',
    'speaker_word_sqrt',
    'word_inverse',
    'uniform_shuffle',
]
SEEDS = [42, 123, 2026]
runs = []
if runs_path.exists():
    runs = pd.read_csv(runs_path).to_dict('records')
completed = {(row['strategy'], int(row['seed'])) for row in runs}
print('resuming completed runs:', len(completed))
for strategy in STRATEGIES:
    for seed in SEEDS:
        if (strategy, seed) in completed:
            print('skip', strategy, seed)
            continue
        runs.append(run_one_seed(strategy, seed))
        pd.DataFrame(runs).to_csv(runs_path, index=False)

runs_df = pd.DataFrame(runs)
runs_df = runs_df[
    runs_df['strategy'].isin(STRATEGIES) & runs_df['seed'].isin(SEEDS)
].copy()
runs_df = runs_df.drop_duplicates(['strategy', 'seed'], keep='last')
runs_df.to_csv(runs_path, index=False)
display(runs_df)
print('TEST SPLIT WAS NOT USED.')


In [ ]:
summary = runs_df.groupby('strategy', sort=False).agg(
    macro_f1_mean=('validation_macro_f1', 'mean'),
    macro_f1_std=('validation_macro_f1', 'std'),
    accuracy_mean=('validation_accuracy', 'mean'),
    balanced_accuracy_mean=('validation_balanced_accuracy', 'mean'),
).reset_index()
baseline = float(
    summary.loc[summary['strategy'] == 'speaker_word_inverse', 'macro_f1_mean'].iloc[0]
)
summary['macro_f1_delta_vs_current'] = summary['macro_f1_mean'] - baseline
display(summary)

pivot = runs_df.pivot(index='seed', columns='strategy', values='validation_macro_f1')
paired_deltas = pivot.subtract(pivot['speaker_word_inverse'], axis=0)
display(paired_deltas)
summary.to_csv(output_dir / 'dataset_v3_sampler_ablation_summary.csv', index=False)
paired_deltas.to_csv(output_dir / 'dataset_v3_sampler_ablation_paired_deltas.csv')

best = summary.sort_values('macro_f1_mean').iloc[-1]
best_strategy = best['strategy']
wins = int((paired_deltas[best_strategy] > 0).sum())
print('Current mean macro-F1:', round(baseline, 4))
print('Best strategy:', best_strategy)
print('Best mean macro-F1:', round(float(best['macro_f1_mean']), 4))
print('Delta:', round(float(best['macro_f1_delta_vs_current']), 4))
print('Wins by seed:', f'{wins}/{len(SEEDS)}')
print('Keep a new strategy only if the mean improves and it wins at least 2/3 seeds.')
print('TEST SPLIT WAS NOT USED.')
